In [0]:
# ===========================================
# Notebook Name:
# 03_identify_deck_fetch_targets
#
# Purpose:
# Identify deck_codes whose deck HTML should
# be (re)fetched. Symmetric to
# 01_identify_result_fetch_targets, but for
# decks instead of tournament results.
#
# Inputs:
# pokemon.silver.tournament_results
# pokemon.bronze.deck_raw
# pokemon.bronze.scrape_log
# pokemon.silver.decks
#
# Output:
# pokemon.ops.deck_fetch_targets
#
# Fetch reasons (priority order):
# 1. manual_refetch (widget override)
# 2. retry_error
# 3. parse_error
# 4. never_fetched
# ===========================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)

DECK_RAW_TABLE = (
    "pokemon.bronze.deck_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

DECKS_TABLE = (
    "pokemon.silver.decks"
)

DECK_FETCH_TARGETS_TABLE = (
    "pokemon.ops.deck_fetch_targets"
)

MAX_FETCH_TARGETS = 500

dbutils.widgets.text(
    "manual_refetch_deck_codes",
    "",
)

manual_refetch_deck_codes = [
    code.strip()
    for code in (
        dbutils.widgets.get(
            "manual_refetch_deck_codes"
        ).split(",")
    )
    if code.strip()
]

print("Output:", DECK_FETCH_TARGETS_TABLE)
print(
    "Manual refetch deck codes:",
    manual_refetch_deck_codes,
)

In [0]:
deck_appearances_df = (
    spark.table(
        TOURNAMENT_RESULTS_TABLE
    )
    .select(
        "deck_code",
        "tournament_id",
        "updated_at",
    )
    .filter(
        F.col("deck_code").isNotNull()
    )
)

latest_appearance_window = (
    Window
    .partitionBy("deck_code")
    .orderBy(
        F.col("updated_at").desc()
    )
)

latest_deck_appearance_df = (
    deck_appearances_df
    .withColumn(
        "row_number",
        F.row_number().over(
            latest_appearance_window
        ),
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .withColumnRenamed(
        "tournament_id",
        "latest_tournament_id",
    )
)

print(
    "Distinct deck_codes in results:",
    latest_deck_appearance_df.count(),
)

In [0]:
if spark.catalog.tableExists(
    DECK_RAW_TABLE
):
    latest_raw_deck_df = (
        spark.table(DECK_RAW_TABLE)
        .groupBy("deck_code")
        .agg(
            F.max(
                "scraped_at"
            ).alias(
                "deck_fetched_at"
            ),
            F.max_by(
                "response_hash",
                "scraped_at",
            ).alias(
                "latest_deck_response_hash"
            ),
        )
        .withColumn(
            "has_raw_deck",
            F.lit(True),
        )
    )

else:
    latest_raw_deck_df = (
        spark.createDataFrame(
            [],
            """
            deck_code STRING,
            deck_fetched_at TIMESTAMP,
            latest_deck_response_hash STRING,
            has_raw_deck BOOLEAN
            """,
        )
    )

print(
    "Deck codes with raw HTML:",
    latest_raw_deck_df.count(),
)

In [0]:
if spark.catalog.tableExists(
    DECKS_TABLE
):
    silver_deck_df = (
        spark.table(DECKS_TABLE)
        .select("deck_code")
        .filter(
            F.col("deck_code").isNotNull()
        )
        .distinct()
        .withColumn(
            "has_silver_deck",
            F.lit(True),
        )
    )

else:
    silver_deck_df = (
        spark.createDataFrame(
            [],
            """
            deck_code STRING,
            has_silver_deck BOOLEAN
            """,
        )
    )

print(
    "Deck codes with Silver rows:",
    silver_deck_df.count(),
)

In [0]:
if spark.catalog.tableExists(
    SCRAPE_LOG_TABLE
):
    latest_log_window = (
        Window
        .partitionBy("deck_code")
        .orderBy(
            F.col("scraped_at").desc()
        )
    )

    latest_deck_log_df = (
        spark.table(SCRAPE_LOG_TABLE)
        .filter(
            F.col("source_type")
            == "deck"
        )
        .withColumnRenamed(
            "source_id",
            "deck_code",
        )
        .withColumn(
            "row_number",
            F.row_number().over(
                latest_log_window
            ),
        )
        .filter(
            F.col("row_number") == 1
        )
        .select(
            "deck_code",
            F.col("status").alias(
                "latest_fetch_status"
            ),
            F.col(
                "error_message"
            ).alias(
                "latest_fetch_error"
            ),
        )
    )

else:
    latest_deck_log_df = (
        spark.createDataFrame(
            [],
            """
            deck_code STRING,
            latest_fetch_status STRING,
            latest_fetch_error STRING
            """,
        )
    )

print(
    "Deck codes with scrape log:",
    latest_deck_log_df.count(),
)

In [0]:
deck_status_df = (
    latest_deck_appearance_df.alias("a")
    .join(
        latest_raw_deck_df.alias("r"),
        on="deck_code",
        how="left",
    )
    .join(
        silver_deck_df.alias("s"),
        on="deck_code",
        how="left",
    )
    .join(
        latest_deck_log_df.alias("l"),
        on="deck_code",
        how="left",
    )
    .withColumn(
        "has_raw_deck",
        F.coalesce(
            F.col("has_raw_deck"),
            F.lit(False),
        ),
    )
    .withColumn(
        "has_silver_deck",
        F.coalesce(
            F.col("has_silver_deck"),
            F.lit(False),
        ),
    )
)

In [0]:
error_status_values = [
    "error",
    "failed",
    "failure",
    "http_error",
]

manual_refetch_set = set(
    manual_refetch_deck_codes
)

deck_targets_base_df = (
    deck_status_df
    .withColumn(
        "fetch_reason",
        F.when(
            F.col("deck_code").isin(
                *manual_refetch_set
            )
            if manual_refetch_set
            else F.lit(False),
            F.lit("manual_refetch"),
        )
        .when(
            F.lower(
                F.coalesce(
                    F.col(
                        "latest_fetch_status"
                    ),
                    F.lit(""),
                )
            ).isin(
                *error_status_values
            ),
            F.lit("retry_error"),
        )
        .when(
            F.col("has_raw_deck")
            & ~F.col(
                "has_silver_deck"
            ),
            F.lit("parse_error"),
        )
        .when(
            ~F.col("has_raw_deck"),
            F.lit("never_fetched"),
        ),
    )
)

deck_fetch_targets_df = (
    deck_targets_base_df
    .filter(
        F.col(
            "fetch_reason"
        ).isNotNull()
    )
)

In [0]:
deck_fetch_targets_df = (
    deck_fetch_targets_df
    .withColumn(
        "priority",
        F.when(
            F.col("fetch_reason")
            == "manual_refetch",
            F.lit(1),
        )
        .when(
            F.col("fetch_reason")
            == "retry_error",
            F.lit(2),
        )
        .when(
            F.col("fetch_reason")
            == "parse_error",
            F.lit(3),
        )
        .when(
            F.col("fetch_reason")
            == "never_fetched",
            F.lit(4),
        )
        .otherwise(F.lit(99)),
    )
)

In [0]:
deck_fetch_targets_df = (
    deck_fetch_targets_df
    .select(
        "deck_code",
        F.col(
            "latest_tournament_id"
        ).alias("tournament_id"),
        "fetch_reason",
        "priority",
        "has_raw_deck",
        "has_silver_deck",
        "deck_fetched_at",
        F.lit(None)
        .cast("timestamp")
        .alias("deck_changed_at"),
        "latest_fetch_status",
        "latest_fetch_error",
        "latest_deck_response_hash",
    )
    .orderBy(
        F.col("priority").asc(),
        F.col("deck_code").desc(),
    )
    .limit(MAX_FETCH_TARGETS)
)

target_count = (
    deck_fetch_targets_df.count()
)

print(
    "Deck fetch target count:",
    target_count,
)

display(deck_fetch_targets_df)

In [0]:
display(
    deck_fetch_targets_df
    .groupBy("fetch_reason")
    .count()
    .orderBy("fetch_reason")
)

In [0]:
duplicate_targets_df = (
    deck_fetch_targets_df
    .groupBy("deck_code")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = (
    duplicate_targets_df.count()
)

if duplicate_count > 0:
    display(duplicate_targets_df)

    raise ValueError(
        f"{duplicate_count} duplicate "
        "deck fetch targets detected"
    )

print(
    "Validation passed: "
    "one target per deck_code"
)

valid_fetch_reasons = [
    "manual_refetch",
    "retry_error",
    "parse_error",
    "never_fetched",
]

invalid_reason_df = (
    deck_fetch_targets_df
    .filter(
        ~F.col(
            "fetch_reason"
        ).isin(*valid_fetch_reasons)
    )
)

if invalid_reason_df.count() > 0:
    display(invalid_reason_df)

    raise ValueError(
        "Invalid fetch reasons detected"
    )

print(
    "Validation passed: "
    "all fetch reasons are valid"
)

In [0]:
(
    deck_fetch_targets_df
    .withColumn(
        "identified_at",
        F.current_timestamp(),
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        DECK_FETCH_TARGETS_TABLE
    )
)

print(
    "Deck fetch targets saved:",
    DECK_FETCH_TARGETS_TABLE,
)

In [0]:
display(
    spark.table(
        DECK_FETCH_TARGETS_TABLE
    )
    .orderBy(
        "priority",
        F.col(
            "identified_at"
        ).desc(),
    )
)